# 06 - Évaluation et métriques

Objectif : calculer les métriques du livrable.

Fichiers concernés : `data/synthetic_cases.csv`, `src/inference.py`, `src/guardrails.py`, `src/metrics.py`, `outputs/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import pandas as pd, time
from pathlib import Path
from src.inference import toy_predict
from src.guardrails import apply_safety_guardrails, validate_prediction

classes = ["normal", "suspected_opacity", "uncertain"]
df = pd.read_csv(DATA_DIR / "synthetic_cases.csv")
rows = []
for _, row in df.iterrows():
    start = time.perf_counter()
    pred = apply_safety_guardrails(toy_predict(PROJECT_ROOT / row["image_path"], mode="baseline"))
    latency_ms = round((time.perf_counter() - start) * 1000, 3)
    valid, errors = validate_prediction(pred)
    rows.append({"case_id": row["case_id"], "filename": Path(row["image_path"]).name, "expected_label": row["label"], "predicted_class": pred["predicted_class"], "confidence": pred["confidence"], "json_valid": valid, "warning_present": bool(pred.get("warning")), "latency_ms": pred.get("latency_ms", latency_ms), "guardrail_errors": ";".join(errors)})
pred_df = pd.DataFrame(rows)
pred_df.to_csv(OUTPUTS_DIR / "evaluation_predictions.csv", index=False)
display(pred_df.head())

In [ ]:
def safe_div(a, b): return a / b if b else 0.0
y_true, y_pred = pred_df["expected_label"], pred_df["predicted_class"]
metrics_summary = pd.DataFrame([{"n": len(pred_df), "accuracy": round((y_true == y_pred).mean(), 4), "json_valid_rate": round(pred_df["json_valid"].mean(), 4), "warning_rate": round(pred_df["warning_present"].mean(), 4), "uncertain_rate": round((y_pred == "uncertain").mean(), 4), "latency_median_ms": round(pred_df["latency_ms"].median(), 4)}])
metrics_summary.to_csv(OUTPUTS_DIR / "metrics_summary.csv", index=False)
confusion = pd.crosstab(y_true, y_pred, rownames=["expected_label"], colnames=["predicted_class"]).reindex(index=classes, columns=classes, fill_value=0)
confusion.to_csv(OUTPUTS_DIR / "confusion_matrix.csv")
per_class, spec = [], []
for klass in classes:
    tp = int(((y_true == klass) & (y_pred == klass)).sum()); fp = int(((y_true != klass) & (y_pred == klass)).sum())
    fn = int(((y_true == klass) & (y_pred != klass)).sum()); tn = int(((y_true != klass) & (y_pred != klass)).sum())
    precision = safe_div(tp, tp + fp); recall = safe_div(tp, tp + fn); f1 = safe_div(2 * precision * recall, precision + recall)
    per_class.append({"class": klass, "tp": tp, "fp": fp, "fn": fn, "precision": round(precision, 4), "recall_sensitivity": round(recall, 4), "f1": round(f1, 4)})
    spec.append({"class": klass, "tn": tn, "fp": fp, "specificity": round(safe_div(tn, tn + fp), 4)})
pd.DataFrame(per_class).to_csv(OUTPUTS_DIR / "per_class_metrics.csv", index=False)
pd.DataFrame(spec).to_csv(OUTPUTS_DIR / "specificity_metrics.csv", index=False)
display(metrics_summary); display(confusion); display(pd.DataFrame(per_class)); display(pd.DataFrame(spec))

Conclusion : transférer précision, rappel/sensibilité, F1, spécificité, confusion matrix et latence médiane dans `src/metrics.py`.